# Bài 1: Tổng Quan Unsupervised Learning

**Session 7 - Advanced Data Science with Python**

---

## Mục tiêu bài học

- Phân biệt **Supervised** vs **Unsupervised** Learning
- Hiểu **4 nhóm thuật toán** chính của Unsupervised Learning
- Nắm vai trò **tiền xử lý & chuẩn hóa** trước khi áp dụng
- Xây dựng pipeline hoàn chỉnh cho bài toán unsupervised

---

## 1. Supervised vs Unsupervised Learning

| Tiêu chí | Supervised | Unsupervised |
|:---|:---|:---|
| **Dữ liệu** | Có nhãn (y) | KHÔNG có nhãn |
| **Mục tiêu** | Dự đoán y từ X | Tìm cấu trúc ẩn trong X |
| **Đánh giá** | Accuracy, F1, RMSE | Silhouette, Inertia, Explained Variance |
| **Ví dụ** | Phân loại spam, dự báo giá | Phân nhóm KH, giảm chiều, phát hiện gian lận |

### Khi nào dùng Unsupervised?

1. **Không có nhãn** (90% dữ liệu thực tế)
2. **Khám phá** cấu trúc ẩn mà ta chưa biết
3. **Tiền xử lý** cho Supervised (feature extraction, dimensionality reduction)
4. **Phát hiện bất thường** (anomaly/fraud detection)

---

## 2. Bốn Nhóm Thuật Toán Unsupervised

| Nhóm | Mục đích | Thuật toán tiêu biểu | Ứng dụng |
|:---|:---|:---|:---|
| **Clustering** | Phân nhóm dữ liệu tương tự | K-Means, DBSCAN, Hierarchical, GMM | Phân nhóm KH, gene |
| **Giảm chiều** | Giảm features, giữ thông tin | PCA, ICA, t-SNE, UMAP | Visualization, feature extraction |
| **Association Rules** | Tìm luật kết hợp | Apriori, FP-Growth | Gợi ý sản phẩm |
| **Anomaly Detection** | Phát hiện bất thường | Isolation Forest, LOF, Autoencoder | Gian lận, lỗi máy |

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100
print('Thư viện sẵn sàng!')

## 3. Tại Sao Phải Chuẩn Hóa Dữ Liệu?

Hầu hết thuật toán unsupervised dùng **khoảng cách** (Euclidean, cosine...).

Nếu feature có scale khác nhau thì feature scale LỚN sẽ **THỐNG TRỊ** khoảng cách.

| Phương pháp | Công thức | Khi nào dùng |
|:---|:---|:---|
| `StandardScaler` | $(x - \mu) / \sigma$ | Phân phối gần normal |
| `MinMaxScaler` | $(x - min) / (max - min)$ | Cần giữ nguyên phân phối |
| `np.log1p()` | $\log(1+x)$ | Dữ liệu skewed (chi tiêu, thu nhập) |

In [ ]:
# Demo: Chuẩn hóa LÀM THAY ĐỔI kết quả clustering
from sklearn.datasets import make_blobs

# Tạo dữ liệu có 2 features scale khác nhau cực đoan
X_raw, y_true = make_blobs(300, centers=3, random_state=42)
X_raw[:, 1] *= 100  # Feature 2 lớn hơn Feature 1 x100 lần

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Không scale
labels_raw = KMeans(3, random_state=42, n_init=10).fit_predict(X_raw)
axes[0].scatter(X_raw[:,0], X_raw[:,1], c=labels_raw, cmap='Set1', s=30)
axes[0].set_title(f'KHONG scale\nSil={silhouette_score(X_raw, labels_raw):.3f}', 
                   fontweight='bold', color='red')

# StandardScaler
X_std = StandardScaler().fit_transform(X_raw)
labels_std = KMeans(3, random_state=42, n_init=10).fit_predict(X_std)
axes[1].scatter(X_std[:,0], X_std[:,1], c=labels_std, cmap='Set1', s=30)
axes[1].set_title(f'StandardScaler\nSil={silhouette_score(X_std, labels_std):.3f}', 
                   fontweight='bold', color='green')

# Ground truth
axes[2].scatter(X_std[:,0], X_std[:,1], c=y_true, cmap='Set1', s=30)
axes[2].set_title('Nhan that (ground truth)', fontweight='bold')

plt.suptitle('Khong chuan hoa -> Ket qua sai!', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 4. Pipeline Tổng Quát Cho Bài Toán Unsupervised

```
Dữ liệu thô
    |
    v
1. EDA & Xử lý missing, outlier
    |
    v
2. Chuẩn hóa (StandardScaler / log)
    |
    v
3. Chọn thuật toán phù hợp
    |
    v
4. Tune parameters (K, eps...)
    |
    v
5. Đánh giá (Silhouette, variance explained...)
    |
    v
6. Phân tích & Diễn giải kết quả
```

In [ ]:
# Demo pipeline với dữ liệu Wholesale Customers
df = pd.read_csv('../Bai thi thu/Course Files/Wholesale customers data.csv')
print(f'Shape: {df.shape}')
print(f'\nMo ta du lieu:')
print(df.describe().round(0).to_string())
print(f'\nMissing: {df.isnull().sum().sum()}')

In [ ]:
# Bước 1: EDA
features = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.ravel(), features):
    ax.hist(df[col], bins=30, edgecolor='black', alpha=0.7)
    ax.set_title(f'{col}\nskew={df[col].skew():.1f}', fontweight='bold')

plt.suptitle('Phan phoi: Tat ca deu RIGHT-SKEWED -> Can log transform', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Bước 2: Log + Scale
X = df[features]
X_log = np.log1p(X)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_log)
print(f'Trung binh sau scale: {X_scaled.mean(axis=0).round(3)}')
print(f'Std sau scale: {X_scaled.std(axis=0).round(3)}')

In [ ]:
# Bước 3-4: K-Means + Tìm K
sil_scores = []
for k in range(2, 8):
    km = KMeans(k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    s = silhouette_score(X_scaled, labels)
    sil_scores.append((k, s))
    print(f'K={k}: Silhouette = {s:.3f}')

best_k = max(sil_scores, key=lambda x: x[1])[0]
print(f'\nK tot nhat: {best_k}')

In [ ]:
# Bước 5-6: Fit + Phân tích
km_final = KMeans(best_k, random_state=42, n_init=10)
df['Cluster'] = km_final.fit_predict(X_scaled)

# Đặc điểm trung bình
print('Chi tieu trung binh theo nhom:')
print(df.groupby('Cluster')[features].mean().round(0).to_string())

# Visualization bằng PCA 2D
X_pca = PCA(2).fit_transform(X_scaled)
plt.figure(figsize=(8, 6))
for c in sorted(df['Cluster'].unique()):
    mask = df['Cluster'] == c
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], label=f'Nhom {c} (n={mask.sum()})', s=40, alpha=0.7)
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.title('Phan nhom Wholesale Customers', fontsize=13, fontweight='bold')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

---

## 5. Bài Thực Hành

Sử dụng **Iris dataset** (150 mẫu, 4 features, 3 loài hoa).

**Yêu cầu** (giả sử KHÔNG biết nhãn):
1. Load data + phân phối features
2. Chuẩn hóa
3. Tìm K tối ưu (Silhouette)
4. K-Means clustering
5. PCA 2D visualization
6. So sánh kết quả clustering với nhãn thật (nếu muốn: `adjusted_rand_score`)

In [ ]:
# =============================================
# BÀI TẬP - Viết code tại đây
# =============================================
from sklearn.datasets import load_iris
from sklearn.metrics import adjusted_rand_score

iris = load_iris()
X_iris = iris.data
y_iris = iris.target  # Dùng để so sánh cuối cùng

# Bước 1: Scale
# X_iris_scaled = ...

# Bước 2: Tìm K tối ưu
# ...

# Bước 3: K-Means
# ...

# Bước 4: PCA 2D + Plot
# ...

# Bước 5: So sánh với nhãn thật
# adjusted_rand_score(y_iris, labels_pred)

---

## TONG HOP: 20% Kien Thuc -> 80% Ung Dung

| # | Kiến thức cốt lõi | Chi tiết |
|:--|:---|:---|
| 1 | **LUÔN chuẩn hóa** trước unsupervised | `StandardScaler()` hoặc `log1p()` + `StandardScaler()` |
| 2 | **4 nhóm thuật toán** | Clustering, Giảm chiều, Association Rules, Anomaly Detection |
| 3 | **Silhouette Score** đánh giá clustering | > 0.5: tốt, < 0.25: kém |
| 4 | **Pipeline chuẩn** | EDA -> Scale -> Fit -> Evaluate -> Interpret |
| 5 | **PCA 2D** để visualize | Luôn dùng PCA chiếu xuống 2D để kiểm tra clustering |

---
**Bài tiếp theo: [Bài 2] K-Means Clustering chi tiết**

# Bài 1: Tổng Quan Unsupervised Learning

**Session 7 - Advanced Data Science with Python**

---

## Mục tiêu bài học

- Phân biệt **Supervised** vs **Unsupervised** Learning
- Hiểu **4 nhóm thuật toán** chính của Unsupervised Learning
- Nắm vai trò **tiền xử lý & chuẩn hóa** trước khi áp dụng
- Xây dựng pipeline hoàn chỉnh cho bài toán unsupervised

---

## 1. Supervised vs Unsupervised Learning

| Tiêu chí | Supervised | Unsupervised |
|:---|:---|:---|
| **Dữ liệu** | Có nhãn (y) | KHÔNG có nhãn |
| **Mục tiêu** | Dự đoán y từ X | Tìm cấu trúc ẩn trong X |
| **Đánh giá** | Accuracy, F1, RMSE | Silhouette, Inertia, Explained Variance |
| **Ví dụ** | Phân loại spam, dự báo giá | Phân nhóm KH, giảm chiều, phát hiện gian lận |

### Khi nào dùng Unsupervised?

1. **Không có nhãn** (90% dữ liệu thực tế)
2. **Khám phá** cấu trúc ẩn mà ta chưa biết
3. **Tiền xử lý** cho Supervised (feature extraction, dimensionality reduction)
4. **Phát hiện bất thường** (anomaly/fraud detection)

---

## 2. Bốn Nhóm Thuật Toán Unsupervised

| Nhóm | Mục đích | Thuật toán tiêu biểu | Ứng dụng |
|:---|:---|:---|:---|
| **Clustering** | Phân nhóm dữ liệu tương tự | K-Means, DBSCAN, Hierarchical, GMM | Phân nhóm KH, gene |
| **Giảm chiều** | Giảm features, giữ thông tin | PCA, ICA, t-SNE, UMAP | Visualization, feature extraction |
| **Association Rules** | Tìm luật kết hợp | Apriori, FP-Growth | Gợi ý sản phẩm |
| **Anomaly Detection** | Phát hiện bất thường | Isolation Forest, LOF, Autoencoder | Gian lận, lỗi máy |

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100
print("✅ Thư viện sẵn sàng!")

## 3. Tại Sao Phải Chuẩn Hóa Dữ Liệu?

Hầu hết thuật toán unsupervised dùng **khoảng cách** (Euclidean, cosine...).

Nếu feature có scale khác nhau → feature scale LỚN sẽ **THỐNG TRỊ** khoảng cách.

| Phương pháp | Công thức | Khi nào dùng |
|:---|:---|:---|
| `StandardScaler` | $(x - \mu) / \sigma$ | Phân phối gần normal |
| `MinMaxScaler` | $(x - min) / (max - min)$ | Cần giữ nguyên phân phối |
| `np.log1p()` | $\log(1+x)$ | Dữ liệu skewed (chi tiêu, thu nhập) |

In [ ]:
# Demo: Chuẩn hóa LÀM THAY ĐỔI kết quả clustering
from sklearn.datasets import make_blobs

# Tạo dữ liệu có 2 features scale khác nhau cực đoan
X_raw, y_true = make_blobs(300, centers=3, random_state=42)
X_raw[:, 1] *= 100  # Feature 2 lớn hơn Feature 1 x100 lần

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Không scale
labels_raw = KMeans(3, random_state=42, n_init=10).fit_predict(X_raw)
axes[0].scatter(X_raw[:,0], X_raw[:,1], c=labels_raw, cmap='Set1', s=30)
axes[0].set_title(f'❌ KHÔNG scale\nSil={silhouette_score(X_raw, labels_raw):.3f}', 
                   fontweight='bold', color='red')

# StandardScaler
X_std = StandardScaler().fit_transform(X_raw)
labels_std = KMeans(3, random_state=42, n_init=10).fit_predict(X_std)
axes[1].scatter(X_std[:,0], X_std[:,1], c=labels_std, cmap='Set1', s=30)
axes[1].set_title(f'✅ StandardScaler\nSil={silhouette_score(X_std, labels_std):.3f}', 
                   fontweight='bold', color='green')

# Ground truth
axes[2].scatter(X_std[:,0], X_std[:,1], c=y_true, cmap='Set1', s=30)
axes[2].set_title('Nhãn thật (ground truth)', fontweight='bold')

plt.suptitle('Không chuẩn hóa → Kết quả sai!', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---

## 4. Pipeline Tổng Quát Cho Bài Toán Unsupervised

```
Dữ liệu thô
    │
    ▼
1. EDA & Xử lý missing, outlier
    │
    ▼
2. Chuẩn hóa (StandardScaler / log)
    │
    ▼
3. Chọn thuật toán phù hợp
    │
    ▼
4. Tune parameters (K, eps...)
    │
    ▼
5. Đánh giá (Silhouette, variance explained...)
    │
    ▼
6. Phân tích & Diễn giải kết quả
```

In [ ]:
# Demo pipeline với dữ liệu Wholesale Customers
df = pd.read_csv('../Bai thi thu/Course Files/Wholesale customers data.csv')
print(f"Shape: {df.shape}")
print(f"\nMô tả dữ liệu:")
print(df.describe().round(0).to_string())
print(f"\nMissing: {df.isnull().sum().sum()}")

In [ ]:
# Bước 1: EDA
features = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.ravel(), features):
    ax.hist(df[col], bins=30, edgecolor='black', alpha=0.7)
    ax.set_title(f'{col}\nskew={df[col].skew():.1f}', fontweight='bold')

plt.suptitle('Phân phối: Tất cả đều RIGHT-SKEWED → Cần log transform', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Bước 2: Log + Scale
X = df[features]
X_log = np.log1p(X)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_log)
print(f"Trung bình sau scale: {X_scaled.mean(axis=0).round(3)}")
print(f"Std sau scale: {X_scaled.std(axis=0).round(3)}")

In [ ]:
# Bước 3-4: K-Means + Tìm K
sil_scores = []
for k in range(2, 8):
    km = KMeans(k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    s = silhouette_score(X_scaled, labels)
    sil_scores.append((k, s))
    print(f"K={k}: Silhouette = {s:.3f}")

best_k = max(sil_scores, key=lambda x: x[1])[0]
print(f"\n🏆 K tốt nhất: {best_k}")

In [ ]:
# Bước 5-6: Fit + Phân tích
km_final = KMeans(best_k, random_state=42, n_init=10)
df['Cluster'] = km_final.fit_predict(X_scaled)

# Đặc điểm trung bình
print("📊 Chi tiêu trung bình theo nhóm:")
print(df.groupby('Cluster')[features].mean().round(0).to_string())

# Visualization bằng PCA 2D
X_pca = PCA(2).fit_transform(X_scaled)
plt.figure(figsize=(8, 6))
for c in sorted(df['Cluster'].unique()):
    mask = df['Cluster'] == c
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], label=f'Nhóm {c} (n={mask.sum()})', s=40, alpha=0.7)
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.title('Phân nhóm Wholesale Customers', fontsize=13, fontweight='bold')
plt.legend(); plt.grid(alpha=0.3); plt.show()

---

## 5. Bài Thực Hành

Sử dụng **Iris dataset** (150 mẫu, 4 features, 3 loài hoa).

**Yêu cầu** (giả sử KHÔNG biết nhãn):
1. Load data + phân phối features
2. Chuẩn hóa
3. Tìm K tối ưu (Silhouette)
4. K-Means clustering
5. PCA 2D visualization
6. So sánh kết quả clustering với nhãn thật (nếu muốn: `adjusted_rand_score`)

In [ ]:
# =============================================
# BÀI TẬP - Viết code tại đây
# =============================================
from sklearn.datasets import load_iris
from sklearn.metrics import adjusted_rand_score

iris = load_iris()
X_iris = iris.data
y_iris = iris.target  # Dùng để so sánh cuối cùng

# Bước 1: Scale
# X_iris_scaled = ...

# Bước 2: Tìm K tối ưu
# ...

# Bước 3: K-Means
# ...

# Bước 4: PCA 2D + Plot
# ...

# Bước 5: So sánh với nhãn thật
# adjusted_rand_score(y_iris, labels_pred)

---

## 📌 TỔNG HỢP: 20% Kiến Thức → 80% Ứng Dụng

| # | Kiến thức cốt lõi | Chi tiết |
|:--|:---|:---|
| 1 | **LUÔN chuẩn hóa** trước unsupervised | `StandardScaler()` hoặc `log1p()` + `StandardScaler()` |
| 2 | **4 nhóm thuật toán** | Clustering, Giảm chiều, Association Rules, Anomaly Detection |
| 3 | **Silhouette Score** đánh giá clustering | > 0.5: tốt, < 0.25: kém |
| 4 | **Pipeline chuẩn** | EDA → Scale → Fit → Evaluate → Interpret |
| 5 | **PCA 2D** để visualize | Luôn dùng PCA chiếu xuống 2D để kiểm tra clustering |

---
**→ Bài tiếp theo: [Bài 2] K-Means Clustering chi tiết**

# Bài 1: Tổng Quan Về Học Không Giám Sát (Unsupervised Learning)

**Session 7 - Advanced Data Science with Python**

---

## Mục tiêu bài học

Sau khi hoàn thành bài này, bạn sẽ:
- Hiểu rõ khái niệm **Học Không Giám Sát** và sự khác biệt với Học Có Giám Sát
- Nắm vững **4 nhóm thuật toán chính** trong Unsupervised Learning
- Biết **khi nào** nên sử dụng từng loại thuật toán
- Nắm được **pipeline** xử lý dữ liệu không giám sát từ đầu đến cuối

---

## 1. Học Không Giám Sát Là Gì?

### 1.1 Định nghĩa

**Học Không Giám Sát (Unsupervised Learning)** là nhánh của Machine Learning trong đó:
- Dữ liệu huấn luyện **KHÔNG có nhãn (label)**
- Mô hình tự **tìm ra cấu trúc ẩn (hidden structure)** trong dữ liệu
- Mục tiêu: khám phá **pattern**, **nhóm**, **mối liên hệ**, hoặc **biểu diễn rút gọn** của dữ liệu

### 1.2 Ví dụ trực giác

Bạn là **nhân viên thư viện mới** nhận 10.000 cuốn sách chưa phân loại:

| Tình huống | Loại học |
|:---|:---|
| Có sẵn danh mục: "Khoa học", "Văn học"... → Gán sách vào danh mục | **Supervised Learning** |
| Không có danh mục → Tự đọc, tìm điểm chung, nhóm sách giống nhau | **Unsupervised Learning** |

### 1.3 So sánh chi tiết

| Tiêu chí | Supervised Learning | Unsupervised Learning |
|:---|:---|:---|
| **Dữ liệu** | Có nhãn (X, y) | Chỉ có features (X) |
| **Mục tiêu** | Dự đoán nhãn | Tìm cấu trúc ẩn |
| **Đánh giá** | Accuracy, F1, MSE | Silhouette, Inertia, Explained Variance |
| **Ví dụ** | Phân loại email spam | Phân nhóm khách hàng |
| **Đặc điểm** | Dễ đánh giá | Cần domain knowledge để đánh giá |

In [ ]:
# === SETUP ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100
print("✅ Thư viện đã sẵn sàng!")

In [ ]:
# Minh họa: Supervised vs Unsupervised
np.random.seed(42)
X, y = make_blobs(n_samples=300, centers=3, cluster_std=1.0, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', s=30, alpha=0.7)
axes[0].set_title('Supervised Learning\n(Dữ liệu CÓ nhãn - màu = nhãn)', fontsize=13)

axes[1].scatter(X[:, 0], X[:, 1], c='gray', s=30, alpha=0.7)
axes[1].set_title('Unsupervised Learning\n(KHÔNG có nhãn → mô hình tự tìm nhóm)', fontsize=13)

plt.tight_layout()
plt.show()
print("→ Unsupervised Learning phải TỰ TÌM ra 3 nhóm mà KHÔNG biết nhãn!")

---

## 2. Bốn Nhóm Thuật Toán Chính

```
Unsupervised Learning
│
├── 1. CLUSTERING (Phân cụm)
│   ├── K-Means          ← Phổ biến nhất, cụm hình cầu
│   ├── DBSCAN           ← Cụm hình dạng bất kỳ, tự tìm số cụm
│   ├── Hierarchical     ← Phân cấp, dendrogram
│   └── GMM              ← Soft clustering (xác suất thuộc mỗi cụm)
│
├── 2. DIMENSIONALITY REDUCTION (Giảm chiều)
│   ├── PCA              ← Giảm chiều tuyến tính, phổ biến nhất
│   ├── ICA              ← Tách nguồn tín hiệu độc lập
│   ├── t-SNE            ← Trực quan hóa phi tuyến
│   └── UMAP             ← Nhanh hơn t-SNE, bảo toàn cấu trúc tốt
│
├── 3. ASSOCIATION RULE LEARNING (Luật kết hợp)
│   ├── Apriori          ← Tìm itemset thường xuyên
│   └── FP-Growth        ← Nhanh hơn Apriori
│
└── 4. ANOMALY DETECTION (Phát hiện bất thường)
    ├── Isolation Forest  ← Nhanh, scale tốt
    ├── LOF              ← Dựa trên mật độ local
    └── One-Class SVM    ← Học boundary bình thường
```

### Bảng tóm tắt ứng dụng

| Nhóm | Câu hỏi trả lời | Ứng dụng điển hình |
|:---|:---|:---|
| **Clustering** | "Dữ liệu nào giống nhau?" | Phân nhóm khách hàng, nén ảnh |
| **Dim. Reduction** | "Thông tin nào quan trọng?" | Trực quan hóa, tăng tốc ML |
| **Association Rules** | "Cái gì hay đi cùng nhau?" | Market basket, gợi ý sản phẩm |
| **Anomaly Detection** | "Cái nào bất thường?" | Gian lận, lỗi máy móc |

In [ ]:
# Minh họa trực quan 4 nhóm thuật toán
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Clustering
X_c, _ = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)
km = KMeans(n_clusters=4, random_state=42, n_init=10)
axes[0,0].scatter(X_c[:,0], X_c[:,1], c=km.fit_predict(X_c), cmap='Set1', s=30)
axes[0,0].scatter(km.cluster_centers_[:,0], km.cluster_centers_[:,1], c='black', marker='X', s=200)
axes[0,0].set_title('1. CLUSTERING\nK-Means phân 4 nhóm tự động', fontsize=13, fontweight='bold')

# 2. Dimensionality Reduction
np.random.seed(42)
X_3d = np.random.randn(200, 3)
X_3d[:, 2] = X_3d[:, 0]*0.5 + X_3d[:, 1]*0.3 + np.random.randn(200)*0.1
X_2d = PCA(n_components=2).fit_transform(X_3d)
axes[0,1].scatter(X_2d[:,0], X_2d[:,1], c=X_3d[:,0], cmap='coolwarm', s=30)
axes[0,1].set_title('2. DIMENSIONALITY REDUCTION\nPCA: 3D → 2D', fontsize=13, fontweight='bold')

# 3. Association Rules
items = ['Bánh mì', 'Bơ', 'Sữa', 'Trứng', 'Phô mai']
matrix = np.array([[1,1,0,1,0],[1,1,1,0,0],[0,0,1,1,1],[1,1,1,1,0],[0,1,1,0,1]])
axes[1,0].imshow(matrix, cmap='YlOrRd', aspect='auto')
axes[1,0].set_xticks(range(5)); axes[1,0].set_xticklabels(items, rotation=45)
axes[1,0].set_yticks(range(5)); axes[1,0].set_yticklabels([f'GD {i+1}' for i in range(5)])
for i in range(5):
    for j in range(5):
        axes[1,0].text(j, i, '✓' if matrix[i,j] else '', ha='center', va='center', fontsize=14)
axes[1,0].set_title('3. ASSOCIATION RULES\nMa trận giao dịch', fontsize=13, fontweight='bold')

# 4. Anomaly Detection
np.random.seed(42)
X_n = np.random.randn(280, 2)*0.5
X_a = np.random.uniform(-3, 3, (20, 2))
X_all = np.vstack([X_n, X_a])
pred = IsolationForest(contamination=0.07, random_state=42).fit_predict(X_all)
axes[1,1].scatter(X_all[pred==1,0], X_all[pred==1,1], c='steelblue', s=30, alpha=0.6, label='Normal')
axes[1,1].scatter(X_all[pred==-1,0], X_all[pred==-1,1], c='red', s=60, marker='x', linewidths=2, label='Anomaly')
axes[1,1].set_title('4. ANOMALY DETECTION\nIsolation Forest', fontsize=13, fontweight='bold')
axes[1,1].legend()

plt.tight_layout()
plt.show()

---

## 3. Tầm Quan Trọng Của Tiền Xử Lý

### 3.1 Tại sao PHẢI chuẩn hóa?

Đây là **SAI LẦM PHỔ BIẾN NHẤT** khi làm Unsupervised Learning:

- Feature A range [0, 100.000] (thu nhập)
- Feature B range [0, 10] (năm kinh nghiệm)  
- → Khoảng cách Euclidean bị **chi phối hoàn toàn** bởi Feature A
- → Feature B gần như **bị bỏ qua** dù nó rất quan trọng

### 3.2 Các phương pháp chuẩn hóa

| Phương pháp | Công thức | Khi nào dùng |
|:---|:---|:---|
| **StandardScaler** | $z = \frac{x - \mu}{\sigma}$ | Mặc định, dữ liệu phân phối gần chuẩn |
| **MinMaxScaler** | $z = \frac{x - x_{min}}{x_{max} - x_{min}}$ | Cần [0, 1], NN, hình ảnh |
| **RobustScaler** | $z = \frac{x - Q2}{Q3 - Q1}$ | Dữ liệu có **nhiều outlier** |

### 3.3 Khoảng cách (Distance Metrics)

| Khoảng cách | Dùng khi |
|:---|:---|
| **Euclidean** | Mặc định cho K-Means, PCA |
| **Manhattan** | Dữ liệu sparse, robust hơn |
| **Cosine** | Text/NLP, so sánh hướng |

In [ ]:
# Demo: Hậu quả của việc KHÔNG chuẩn hóa
np.random.seed(42)
income = np.concatenate([np.random.normal(15, 3, 100), np.random.normal(50, 5, 100)])
experience = np.concatenate([np.random.normal(2, 0.5, 100), np.random.normal(8, 1, 100)])
X_raw = np.column_stack([income, experience])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

labels_raw = KMeans(2, random_state=42, n_init=10).fit_predict(X_raw)
axes[0].scatter(X_raw[:,0], X_raw[:,1], c=labels_raw, cmap='Set1', s=30)
axes[0].set_xlabel('Thu nhập (triệu)'); axes[0].set_ylabel('Năm KN')
axes[0].set_title('❌ KHÔNG chuẩn hóa\n(Thu nhập chi phối → sai!)', color='red', fontsize=13, fontweight='bold')

X_sc = StandardScaler().fit_transform(X_raw)
labels_sc = KMeans(2, random_state=42, n_init=10).fit_predict(X_sc)
axes[1].scatter(X_raw[:,0], X_raw[:,1], c=labels_sc, cmap='Set1', s=30)
axes[1].set_xlabel('Thu nhập (triệu)'); axes[1].set_ylabel('Năm KN')
axes[1].set_title('✅ CÓ chuẩn hóa\n(Cả 2 features công bằng)', color='green', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()
print("📌 QUY TẮC VÀNG: LUÔN StandardScaler() trước khi dùng bất kỳ Unsupervised algorithm nào!")

---

## 4. Pipeline Tổng Quát

```
1. Thu thập dữ liệu
        ↓
2. EDA (Exploratory Data Analysis)
        ↓
3. Tiền xử lý
   ├── Xử lý missing values
   ├── Encoding categorical
   └── Scaling ← BẮT BUỘC
        ↓
4. Chọn thuật toán phù hợp
        ↓
5. Huấn luyện & Điều chỉnh tham số
        ↓
6. Đánh giá kết quả (metrics + visualization)
        ↓
7. Diễn giải & Quyết định kinh doanh
```

In [ ]:
# === DEMO PIPELINE HOÀN CHỈNH: Wholesale Customers ===

# Bước 1-2: Load & EDA
df = pd.read_csv('../Bai thi thu/Course Files/Wholesale customers data.csv')
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print("\n→ Channel: 1=Hotel/Restaurant, 2=Retail")
print("→ Fresh~Delicassen: Chi tiêu hàng năm")
df.describe().round(0)

In [ ]:
# Bước 3: Tiền xử lý
features = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']
X_scaled = StandardScaler().fit_transform(df[features])
print(f"Missing: {df[features].isnull().sum().sum()}")
print("✅ Đã chuẩn hóa StandardScaler")

In [ ]:
# Bước 4-6: K-Means + Đánh giá
results = []
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, labels)
    results.append({'K': k, 'Inertia': km.inertia_, 'Silhouette': sil})

res = pd.DataFrame(results)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(res['K'], res['Inertia'], 'bo-', lw=2)
axes[0].set_title('Elbow Method', fontweight='bold'); axes[0].set_xlabel('K'); axes[0].grid(alpha=0.3)
axes[1].plot(res['K'], res['Silhouette'], 'rs-', lw=2)
axes[1].set_title('Silhouette Score (cao = tốt)', fontweight='bold'); axes[1].set_xlabel('K'); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"\n🏆 K tối ưu: {int(res.loc[res['Silhouette'].idxmax(), 'K'])}")

In [ ]:
# Bước 7: Diễn giải
df['Cluster'] = KMeans(2, random_state=42, n_init=10).fit_predict(X_scaled)
print("Đặc điểm trung bình từng nhóm:")
print(df.groupby('Cluster')[features].mean().round(0).to_string())

X_pca = PCA(2).fit_transform(X_scaled)
plt.figure(figsize=(10, 6))
for c in [0, 1]:
    m = df['Cluster'] == c
    plt.scatter(X_pca[m,0], X_pca[m,1], label=f'Nhóm {c} (n={m.sum()})', s=40, alpha=0.7)
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.title('Phân nhóm khách hàng Wholesale', fontsize=14, fontweight='bold')
plt.legend(); plt.grid(alpha=0.3); plt.show()

print("\n📊 DIỄN GIẢI:")
print("  Nhóm 0: Chi cao Fresh, Frozen → Nhà hàng / Quán ăn")
print("  Nhóm 1: Chi cao Milk, Grocery, Detergents → Siêu thị bán lẻ")

---

## 5. Bài Thực Hành: Iris Dataset

**Yêu cầu**: Load Iris (không nhãn) → Chuẩn hóa → K-Means K=3 → So sánh với nhãn thật → PCA 2D

In [ ]:
# BÀI GIẢI MẪU
from sklearn.datasets import load_iris
from sklearn.metrics import adjusted_rand_score

iris = load_iris()
X_iris = StandardScaler().fit_transform(iris.data)
y_pred = KMeans(3, random_state=42, n_init=10).fit_predict(X_iris)

ari = adjusted_rand_score(iris.target, y_pred)
print(f"Adjusted Rand Index: {ari:.4f} (1.0 = hoàn hảo)")

X_pca = PCA(2).fit_transform(X_iris)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, name in enumerate(iris.target_names):
    m = iris.target == i
    axes[0].scatter(X_pca[m,0], X_pca[m,1], label=name, s=40, alpha=0.7)
axes[0].set_title('Nhãn thật', fontweight='bold'); axes[0].legend()

for i in range(3):
    m = y_pred == i
    axes[1].scatter(X_pca[m,0], X_pca[m,1], label=f'Cluster {i}', s=40, alpha=0.7)
axes[1].set_title(f'K-Means (ARI={ari:.3f})', fontweight='bold'); axes[1].legend()
plt.tight_layout(); plt.show()

print("✅ K-Means tìm nhóm gần giống nhãn thật MÀ KHÔNG CẦN nhãn!")

---

## 📌 TỔNG HỢP: 20% Kiến Thức → 80% Ứng Dụng

| # | Kiến thức cốt lõi | Tại sao quan trọng |
|:--|:---|:---|
| 1 | **LUÔN `StandardScaler()` trước** | 90% lỗi clustering đến từ không chuẩn hóa |
| 2 | **Chọn đúng thuật toán** cho đúng bài toán | Clustering ≠ PCA ≠ Apriori ≠ Anomaly |
| 3 | **Silhouette Score** là metric #1 cho clustering | > 0.5 tốt, > 0.7 rất tốt, < 0.25 kém |
| 4 | **Không có đáp án đúng** trong unsupervised | Cần kết hợp metrics + vis + domain knowledge |
| 5 | **Pipeline DỨT KHOÁT**: EDA → Scale → Model → Evaluate | Bỏ bước nào cũng sai |

### Cheat-sheet: Khi nào dùng gì?

| Bài toán | Thuật toán đầu tiên | Bài học |
|:---|:---|:---|
| Phân nhóm khách hàng | K-Means | Bài 2 |
| Giảm features / Trực quan hóa | PCA | Bài 3 |
| Tách tín hiệu trộn lẫn | ICA | Bài 4 |
| Tìm sản phẩm mua kèm | Apriori | Bài 5 |
| Phát hiện gian lận | Isolation Forest | Bài 6 |

---
**→ Bài tiếp theo: [Bài 2] K-Means Clustering**

# Bài 1: Tổng Quan Về Học Không Giám Sát (Unsupervised Learning)

**Session 7 - Advanced Data Science with Python**

---

## Mục tiêu bài học

Sau khi hoàn thành bài học này, bạn sẽ:
- Hiểu rõ khái niệm **Học Không Giám Sát** và sự khác biệt với Học Có Giám Sát
- Nắm vững các **nhóm thuật toán chính** trong Unsupervised Learning
- Hiểu **khi nào** nên sử dụng từng loại thuật toán
- Có cái nhìn tổng thể về **pipeline** xử lý dữ liệu không giám sát

---

## 1. Học Không Giám Sát là gì?

### 1.1 Định nghĩa

**Học Không Giám Sát (Unsupervised Learning)** là nhánh của Machine Learning, trong đó:
- Dữ liệu huấn luyện **KHÔNG có nhãn (label)**
- Mô hình tự **tìm ra cấu trúc ẩn (hidden structure)** trong dữ liệu
- Mục tiêu: khám phá **pattern**, **nhóm**, **mối liên hệ**, hoặc **biểu diễn rút gọn** của dữ liệu

### 1.2 Ví dụ thực tế đời thường

Hãy tưởng tượng bạn là một **nhân viên thư viện mới** nhận được 10.000 cuốn sách chưa được phân loại:

| Tình huống | Loại học |
|:---|:---|
| Có sẵn danh mục: "Khoa học", "Văn học", "Lịch sử"... → Gán từng sách vào danh mục | **Supervised Learning** |
| Không có danh mục → Tự đọc, tìm điểm chung, nhóm sách giống nhau lại | **Unsupervised Learning** |

### 1.3 So sánh chi tiết

| Tiêu chí | Supervised Learning | Unsupervised Learning |
|:---|:---|:---|
| **Dữ liệu** | Có nhãn (X, y) | Chỉ có features (X) |
| **Mục tiêu** | Dự đoán nhãn cho dữ liệu mới | Tìm cấu trúc ẩn |
| **Đánh giá** | Accuracy, F1, MSE... | Silhouette, Inertia, Explained Variance... |
| **Ví dụ** | Phân loại email spam | Phân nhóm khách hàng |
| **Độ khó** | Dễ đánh giá hơn | Khó đánh giá, cần domain knowledge |

In [ ]:
# Minh họa sự khác biệt bằng code
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_classification
import warnings
warnings.filterwarnings('ignore')

# Tạo dữ liệu
np.random.seed(42)
X, y = make_blobs(n_samples=300, centers=3, cluster_std=1.0, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Supervised: Có nhãn
axes[0].scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', s=30, alpha=0.7)
axes[0].set_title('Supervised Learning\n(Dữ liệu CÓ nhãn - màu sắc = nhãn)', fontsize=13)
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')

# Unsupervised: Không có nhãn
axes[1].scatter(X[:, 0], X[:, 1], c='gray', s=30, alpha=0.7)
axes[1].set_title('Unsupervised Learning\n(Dữ liệu KHÔNG có nhãn - tất cả cùng màu)', fontsize=13)
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')

plt.tight_layout()
plt.show()
print("→ Trong Unsupervised Learning, mô hình phải TỰ TÌM ra 3 nhóm này!")

---

## 2. Các Nhóm Thuật Toán Chính Trong Unsupervised Learning

Unsupervised Learning chia thành **4 nhóm lớn**:

```
Unsupervised Learning
│
├── 1. CLUSTERING (Phân cụm)
│   ├── K-Means
│   ├── Hierarchical Clustering
│   ├── DBSCAN
│   └── Gaussian Mixture Models (GMM)
│
├── 2. DIMENSIONALITY REDUCTION (Giảm chiều)
│   ├── PCA (Principal Component Analysis)
│   ├── ICA (Independent Component Analysis)
│   ├── t-SNE
│   └── UMAP
│
├── 3. ASSOCIATION RULE LEARNING (Luật kết hợp)
│   ├── Apriori
│   ├── FP-Growth
│   └── Eclat
│
└── 4. ANOMALY DETECTION (Phát hiện bất thường)
    ├── Isolation Forest
    ├── Local Outlier Factor (LOF)
    ├── One-Class SVM
    └── Autoencoder
```

### 2.1 Clustering - Phân cụm

**Mục tiêu**: Nhóm các điểm dữ liệu tương tự vào cùng một cụm.

**Ứng dụng thực tế**:
- 🛒 Phân nhóm khách hàng theo hành vi mua sắm
- 📰 Nhóm các bài báo theo chủ đề
- 🏥 Phân loại bệnh nhân theo triệu chứng
- 🖼️ Nén ảnh (color quantization)

### 2.2 Dimensionality Reduction - Giảm chiều dữ liệu

**Mục tiêu**: Giảm số chiều (features) trong khi giữ lại nhiều thông tin nhất có thể.

**Ứng dụng thực tế**:
- 📊 Trực quan hóa dữ liệu nhiều chiều
- ⚡ Tăng tốc độ huấn luyện mô hình
- 🔊 Tách nguồn tín hiệu (ICA - Cocktail Party Problem)
- 🧬 Phân tích gen (genomics)

### 2.3 Association Rule Learning - Luật kết hợp

**Mục tiêu**: Tìm mối quan hệ giữa các item trong tập dữ liệu giao dịch.

**Ứng dụng thực tế**:
- 🛍️ Market Basket Analysis: "Khách mua bánh mì thường mua thêm bơ"
- 💊 Phân tích đơn thuốc
- 🌐 Gợi ý sản phẩm (Netflix, Amazon)

### 2.4 Anomaly Detection - Phát hiện bất thường

**Mục tiêu**: Phát hiện các điểm dữ liệu khác biệt đáng kể so với phần còn lại.

**Ứng dụng thực tế**:
- 💳 Phát hiện gian lận thẻ tín dụng
- 🏭 Giám sát máy móc công nghiệp
- 🔒 An ninh mạng (phát hiện xâm nhập)
- 🏥 Phát hiện bệnh hiếm

In [ ]:
# Minh họa trực quan 4 nhóm thuật toán
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# --- 1. Clustering ---
X_cluster, _ = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_cluster)
axes[0, 0].scatter(X_cluster[:, 0], X_cluster[:, 1], c=labels, cmap='Set1', s=30, alpha=0.7)
axes[0, 0].scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], 
                    c='black', marker='X', s=200, edgecolors='white', linewidths=2)
axes[0, 0].set_title('1. CLUSTERING\n(K-Means phân 4 nhóm)', fontsize=13, fontweight='bold')

# --- 2. Dimensionality Reduction ---
np.random.seed(42)
X_3d = np.random.randn(200, 3)
X_3d[:, 2] = X_3d[:, 0] * 0.5 + X_3d[:, 1] * 0.3 + np.random.randn(200) * 0.1
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_3d)
axes[0, 1].scatter(X_2d[:, 0], X_2d[:, 1], c=X_3d[:, 0], cmap='coolwarm', s=30, alpha=0.7)
axes[0, 1].set_title(f'2. DIMENSIONALITY REDUCTION\n(3D → 2D, giữ {pca.explained_variance_ratio_.sum()*100:.1f}% thông tin)', 
                      fontsize=13, fontweight='bold')
axes[0, 1].set_xlabel('PC1')
axes[0, 1].set_ylabel('PC2')

# --- 3. Association Rules ---
items = ['Bánh mì', 'Bơ', 'Sữa', 'Trứng', 'Phô mai']
matrix = np.array([
    [1, 1, 0, 1, 0],
    [1, 1, 1, 0, 0],
    [0, 0, 1, 1, 1],
    [1, 1, 1, 1, 0],
    [0, 1, 1, 0, 1],
])
axes[1, 0].imshow(matrix, cmap='YlOrRd', aspect='auto')
axes[1, 0].set_xticks(range(5))
axes[1, 0].set_xticklabels(items, rotation=45)
axes[1, 0].set_yticks(range(5))
axes[1, 0].set_yticklabels([f'GD {i+1}' for i in range(5)])
axes[1, 0].set_title('3. ASSOCIATION RULES\n(Ma trận giao dịch)', fontsize=13, fontweight='bold')
for i in range(5):
    for j in range(5):
        axes[1, 0].text(j, i, '✓' if matrix[i,j] else '', ha='center', va='center', fontsize=14)

# --- 4. Anomaly Detection ---
np.random.seed(42)
X_normal = np.random.randn(280, 2) * 0.5
X_anomaly = np.random.uniform(low=-3, high=3, size=(20, 2))
X_all = np.vstack([X_normal, X_anomaly])
iso = IsolationForest(contamination=0.07, random_state=42)
pred = iso.fit_predict(X_all)
axes[1, 1].scatter(X_all[pred == 1, 0], X_all[pred == 1, 1], c='steelblue', s=30, alpha=0.6, label='Bình thường')
axes[1, 1].scatter(X_all[pred == -1, 0], X_all[pred == -1, 1], c='red', s=60, marker='x', linewidths=2, label='Bất thường')
axes[1, 1].set_title('4. ANOMALY DETECTION\n(Phát hiện điểm bất thường)', fontsize=13, fontweight='bold')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

---

## 3. Các Khái Niệm Toán Học Cốt Lõi

Để hiểu sâu Unsupervised Learning, cần nắm vững các khái niệm sau:

### 3.1 Khoảng cách (Distance Metrics)

Hầu hết các thuật toán Unsupervised Learning đều dựa trên **khái niệm khoảng cách** giữa các điểm dữ liệu.

| Khoảng cách | Công thức | Đặc điểm |
|:---|:---|:---|
| **Euclidean** | $d = \sqrt{\sum_{i=1}^{n}(x_i - y_i)^2}$ | Phổ biến nhất, nhạy cảm với scale |
| **Manhattan** | $d = \sum_{i=1}^{n}|x_i - y_i|$ | Robust hơn với outlier |
| **Cosine** | $d = 1 - \frac{\vec{x} \cdot \vec{y}}{||\vec{x}|| \cdot ||\vec{y}||}$ | Đo hướng, không phụ thuộc độ lớn |

### 3.2 Ma trận hiệp phương sai (Covariance Matrix)

Nền tảng cho PCA và nhiều thuật toán khác:

$$\text{Cov}(X, Y) = \frac{1}{n-1}\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})$$

### 3.3 Entropy và Information Theory

Cơ sở cho ICA và một số phương pháp clustering:

$$H(X) = -\sum_{i} p(x_i) \log p(x_i)$$

In [ ]:
# Minh họa: So sánh các loại khoảng cách
from scipy.spatial.distance import euclidean, cityblock, cosine

# Hai vector ví dụ
A = np.array([1, 2])
B = np.array([4, 6])

print("=" * 50)
print("So sánh các loại khoảng cách")
print("=" * 50)
print(f"Điểm A: {A}")
print(f"Điểm B: {B}")
print(f"\nEuclidean Distance: {euclidean(A, B):.4f}")
print(f"Manhattan Distance:  {cityblock(A, B):.4f}")
print(f"Cosine Distance:     {cosine(A, B):.4f}")

# Trực quan hóa
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, title in zip(axes, ['Euclidean (đường chim bay)', 'Manhattan (đường phố)', 'Cosine (góc giữa 2 vector)']):
    ax.plot(*A, 'ro', markersize=10, label='A(1,2)')
    ax.plot(*B, 'bs', markersize=10, label='B(4,6)')
    ax.set_xlim(-0.5, 6)
    ax.set_ylim(-0.5, 8)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

# Euclidean: đường thẳng
axes[0].plot([A[0], B[0]], [A[1], B[1]], 'g-', linewidth=2)
axes[0].annotate(f'd = {euclidean(A, B):.2f}', xy=(2.5, 4.5), fontsize=12, color='green')

# Manhattan: đường vuông góc
axes[1].plot([A[0], B[0], B[0]], [A[1], A[1], B[1]], 'g-', linewidth=2)
axes[1].annotate(f'd = {cityblock(A, B):.0f}', xy=(3, 3), fontsize=12, color='green')

# Cosine: góc
axes[2].plot([0, A[0]], [0, A[1]], 'r-', linewidth=2)
axes[2].plot([0, B[0]], [0, B[1]], 'b-', linewidth=2)
theta = np.linspace(np.arctan2(A[1], A[0]), np.arctan2(B[1], B[0]), 30)
axes[2].plot(1.5*np.cos(theta), 1.5*np.sin(theta), 'g-', linewidth=2)
axes[2].annotate(f'cosine dist = {cosine(A, B):.4f}', xy=(1, 5), fontsize=11, color='green')

plt.tight_layout()
plt.show()

---

## 4. Tầm Quan Trọng Của Tiền Xử Lý Dữ Liệu

Trong Unsupervised Learning, **tiền xử lý (preprocessing)** cực kỳ quan trọng vì:

### ⚠️ Vì sao phải Chuẩn hóa (Scaling)?

- Nếu Feature A có range [0, 1000] và Feature B có range [0, 1]
- Khoảng cách Euclidean sẽ bị **chi phối hoàn toàn** bởi Feature A
- Feature B gần như **bị bỏ qua** → kết quả sai lệch

### Các phương pháp chuẩn hóa phổ biến:

| Phương pháp | Công thức | Khi nào dùng |
|:---|:---|:---|
| **StandardScaler** | $z = \frac{x - \mu}{\sigma}$ | Dữ liệu phân phối gần chuẩn |
| **MinMaxScaler** | $z = \frac{x - x_{min}}{x_{max} - x_{min}}$ | Cần giá trị trong [0, 1] |
| **RobustScaler** | $z = \frac{x - Q2}{Q3 - Q1}$ | Dữ liệu có nhiều outlier |

In [ ]:
# Minh họa: Tác động của Scaling đến K-Means Clustering
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Tạo dữ liệu với scale khác nhau
np.random.seed(42)
# Feature 1: Thu nhập (đơn vị: triệu VND) - range lớn
income = np.concatenate([np.random.normal(15, 3, 100), np.random.normal(50, 5, 100)])
# Feature 2: Số năm kinh nghiệm - range nhỏ
experience = np.concatenate([np.random.normal(2, 0.5, 100), np.random.normal(8, 1, 100)])

X_raw = np.column_stack([income, experience])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Không chuẩn hóa
km_raw = KMeans(n_clusters=2, random_state=42, n_init=10)
labels_raw = km_raw.fit_predict(X_raw)
axes[0].scatter(X_raw[:, 0], X_raw[:, 1], c=labels_raw, cmap='Set1', s=30, alpha=0.7)
axes[0].set_xlabel('Thu nhập (triệu VND)', fontsize=11)
axes[0].set_ylabel('Năm kinh nghiệm', fontsize=11)
axes[0].set_title('❌ KHÔNG chuẩn hóa\n(Thu nhập chi phối hoàn toàn)', fontsize=13, fontweight='bold', color='red')

# Có chuẩn hóa
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
km_scaled = KMeans(n_clusters=2, random_state=42, n_init=10)
labels_scaled = km_scaled.fit_predict(X_scaled)
axes[1].scatter(X_raw[:, 0], X_raw[:, 1], c=labels_scaled, cmap='Set1', s=30, alpha=0.7)
axes[1].set_xlabel('Thu nhập (triệu VND)', fontsize=11)
axes[1].set_ylabel('Năm kinh nghiệm', fontsize=11)
axes[1].set_title('✅ CÓ chuẩn hóa (StandardScaler)\n(Cả 2 features đều được xét công bằng)', fontsize=13, fontweight='bold', color='green')

plt.tight_layout()
plt.show()

print("\n📌 KẾT LUẬN: Luôn chuẩn hóa dữ liệu trước khi áp dụng Unsupervised Learning!")

---

## 5. Pipeline Tổng Quát Cho Unsupervised Learning

```
1. Thu thập dữ liệu
        ↓
2. Khám phá dữ liệu (EDA)
        ↓
3. Tiền xử lý
   ├── Xử lý missing values
   ├── Encoding (nếu có categorical)
   └── Scaling / Normalization
        ↓
4. Chọn thuật toán
   ├── Clustering → K-Means, DBSCAN...
   ├── Giảm chiều → PCA, ICA...
   ├── Luật kết hợp → Apriori...
   └── Anomaly → Isolation Forest...
        ↓
5. Huấn luyện mô hình
        ↓
6. Đánh giá kết quả
   ├── Metrics nội tại (Silhouette, Inertia...)
   ├── Trực quan hóa
   └── Domain knowledge
        ↓
7. Diễn giải & hành động
```

In [ ]:
# Ví dụ: Pipeline đầy đủ sử dụng dữ liệu Wholesale Customers
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Bước 1: Thu thập dữ liệu
df = pd.read_csv('../Bai thi thu/Course Files/Wholesale customers data.csv')
print("=" * 60)
print("BƯỚC 1: Khám phá dữ liệu")
print("=" * 60)
print(f"\nKích thước: {df.shape}")
print(f"\nCác cột: {list(df.columns)}")
print(f"\nMô tả thống kê:")
df.describe().round(1)

In [ ]:
# Bước 2 & 3: EDA + Tiền xử lý
print("=" * 60)
print("BƯỚC 2 & 3: Tiền xử lý")
print("=" * 60)

# Chọn features số (bỏ Channel, Region)
features = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']
X = df[features]

print(f"\nMissing values:\n{X.isnull().sum()}")
print(f"\nTrước chuẩn hóa - Mean: {X.mean().values.round(0)}")
print(f"Trước chuẩn hóa - Std:  {X.std().values.round(0)}")

# Chuẩn hóa
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"\nSau chuẩn hóa - Mean: {X_scaled.mean(axis=0).round(4)}")
print(f"Sau chuẩn hóa - Std:  {X_scaled.std(axis=0).round(4)}")
print("\n✅ Dữ liệu đã sẵn sàng!")

In [ ]:
# Bước 4, 5, 6: Chọn thuật toán → Huấn luyện → Đánh giá
print("=" * 60)
print("BƯỚC 4-6: Huấn luyện & Đánh giá")
print("=" * 60)

# Thử nhiều số cụm khác nhau
results = []
K_range = range(2, 8)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, labels)
    results.append({'K': k, 'Inertia': km.inertia_, 'Silhouette': sil})
    print(f"  K={k}: Inertia={km.inertia_:.1f}, Silhouette={sil:.4f}")

results_df = pd.DataFrame(results)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow Method
axes[0].plot(results_df['K'], results_df['Inertia'], 'bo-', linewidth=2)
axes[0].set_xlabel('Số cụm K')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Silhouette Score
axes[1].plot(results_df['K'], results_df['Silhouette'], 'rs-', linewidth=2)
axes[1].set_xlabel('Số cụm K')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score (càng cao càng tốt)', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

best_k = results_df.loc[results_df['Silhouette'].idxmax(), 'K']
print(f"\n🏆 Số cụm tối ưu (theo Silhouette): K = {int(best_k)}")

In [ ]:
# Bước 7: Diễn giải kết quả
print("=" * 60)
print("BƯỚC 7: Diễn giải kết quả")
print("=" * 60)

km_final = KMeans(n_clusters=2, random_state=42, n_init=10)
df['Cluster'] = km_final.fit_predict(X_scaled)

# Phân tích đặc điểm từng cụm
cluster_summary = df.groupby('Cluster')[features].mean().round(0)
print("\nĐặc điểm trung bình từng cụm:")
print(cluster_summary.to_string())

print(f"\nSố lượng mỗi cụm:")
print(df['Cluster'].value_counts().sort_index())

# Trực quan hóa bằng PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(10, 6))
for cluster in sorted(df['Cluster'].unique()):
    mask = df['Cluster'] == cluster
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], 
                label=f'Cụm {cluster} (n={mask.sum()})', s=40, alpha=0.7)

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.title('Kết quả phân cụm khách hàng (Wholesale Customers)', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

print("\n📊 DIỄN GIẢI:")
print("- Cụm 0: Khách hàng chi tiêu CAO cho Fresh, Frozen → Nhà hàng, quán ăn")
print("- Cụm 1: Khách hàng chi tiêu CAO cho Milk, Grocery, Detergents → Siêu thị, cửa hàng bán lẻ")

---

## 6. Bài Thực Hành

### Bài tập: Khám phá dữ liệu Iris bằng Unsupervised Learning

**Yêu cầu**:
1. Load dataset Iris (từ sklearn)
2. Chuẩn hóa dữ liệu
3. Áp dụng K-Means với K=3
4. So sánh kết quả clustering với nhãn thật
5. Trực quan hóa kết quả bằng PCA 2D

In [ ]:
# BÀI GIẢI MẪU

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score

# 1. Load data
iris = load_iris()
X_iris = iris.data
y_true = iris.target
print(f"Iris dataset: {X_iris.shape[0]} mẫu, {X_iris.shape[1]} features")
print(f"Features: {iris.feature_names}")
print(f"Classes: {iris.target_names}")

# 2. Chuẩn hóa
scaler = StandardScaler()
X_iris_scaled = scaler.fit_transform(X_iris)

# 3. K-Means
km = KMeans(n_clusters=3, random_state=42, n_init=10)
y_pred = km.fit_predict(X_iris_scaled)

# 4. So sánh
ari = adjusted_rand_score(y_true, y_pred)
print(f"\nAdjusted Rand Index: {ari:.4f} (1.0 = hoàn hảo)")

# 5. Trực quan hóa
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_iris_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Nhãn thật
for i, name in enumerate(iris.target_names):
    mask = y_true == i
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], label=name, s=40, alpha=0.7)
axes[0].set_title('Nhãn thật (Ground Truth)', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')

# K-Means clusters
for i in range(3):
    mask = y_pred == i
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], label=f'Cluster {i}', s=40, alpha=0.7)
axes[1].set_title(f'K-Means Clustering (ARI={ari:.3f})', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')

plt.tight_layout()
plt.show()

print("\n✅ K-Means có thể tìm ra các nhóm gần giống nhãn thật MÀ KHÔNG CẦN nhãn!")

---

## 📌 TỔNG HỢP: 20% Kiến Thức → 80% Ứng Dụng

### Quy tắc Pareto cho Unsupervised Learning:

| # | Kiến thức cốt lõi | Tại sao quan trọng |
|:--|:---|:---|
| 1 | **LUÔN chuẩn hóa dữ liệu** trước khi áp dụng bất kỳ thuật toán nào | 90% lỗi trong Unsupervised Learning đến từ việc không chuẩn hóa |
| 2 | **Chọn đúng loại thuật toán** cho đúng vấn đề | Clustering ≠ Giảm chiều ≠ Luật kết hợp ≠ Phát hiện bất thường |
| 3 | **Hiểu khoảng cách Euclidean** | Nền tảng của K-Means, Hierarchical Clustering, và nhiều thuật toán khác |
| 4 | **Ma trận hiệp phương sai** | Nền tảng của PCA - thuật toán giảm chiều phổ biến nhất |
| 5 | **Không có "đáp án đúng"** | Unsupervised Learning cần thử nhiều cách + domain knowledge để đánh giá |

### Bảng cheat-sheet: Khi nào dùng gì?

| Bài toán | Thuật toán đầu tiên nên thử | Bài học chi tiết |
|:---|:---|:---|
| Phân nhóm khách hàng | **K-Means** | Bài 2 |
| Giảm features trước khi train model | **PCA** | Bài 3 |
| Tách tín hiệu trộn lẫn | **ICA** | Bài 4 |
| Tìm sản phẩm mua kèm | **Apriori** | Bài 5 |
| Phát hiện giao dịch gian lận | **Isolation Forest** | Bài 6 |

---

**→ Bài tiếp theo: [Bài 2] K-Means Clustering từ Scratch bằng NumPy**